# Customer Churn Prediction

Original modelling code, organized in execution order. Run from top to bottom with the [pinned environment](../requirements.txt).

[Load data](#1-load-and-inspect-the-dataset) · [Clean data](#2-target-split-and-dataset-cleaning) · [Encode and scale](#3-encode-and-scale-features) · [Baselines](#4-baseline-models) · [Oversampling](#5-random-oversampling) · [Tune LR](#6-hyperparameter-search) · [Threshold](#7-threshold-selection) · [Evaluate](#8-optimized-model-evaluation)

The original statements and their order are preserved. Duplicate preliminary preparation and the package-install command are omitted; dependencies are in `requirements.txt`.

**Evaluation is exploratory:** the test set was reused, OHE was fitted before CV, and model and threshold selection reuse the same folds. See [Project Documentation](../Project%20Documentation/README.md) for results and limits.

Code developed with assistance from **CODEX**.

## 1. Load and inspect the dataset

IBM Telco Customer Churn: 7,043 records. The original inspection identifies 11 blank `TotalCharges`, all with `tenure=0`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df.head()

In [ ]:
# Contar filas duplicadas
print("Filas duplicadas:", df.duplicated().sum())

# Mostrar las filas duplicadas, si existen
df[df.duplicated(keep=False)]

In [ ]:
# Mostrar las filas con valores vacíos ocultos en TotalCharges
df[df["TotalCharges"].astype(str).str.strip() == ""]

## 2. Target, split and dataset cleaning

Exclude `customerID`, encode `Churn` as `No=0` / `Yes=1`, then create the stratified 80/20 split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

# 1. Separar variables predictoras y variable objetivo
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

# 2. Dividir en entrenamiento (80%) y test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



### Clean `TotalCharges`

Convert to numeric and fill the empty charges with zero, separately in training and test. These empty values correspond to customers with zero tenure.

In [ ]:
# 3. Convertir TotalCharges e imputar valores vacíos con 0
for dataset in [X_train, X_test]:
    dataset["TotalCharges"] = pd.to_numeric(
        dataset["TotalCharges"], errors="coerce"
    ).fillna(0)



## 3. Encode and scale features

Identify feature types, fit One-Hot Encoding on training data and apply the same encoder to test data.

In [ ]:
# 4. Identificar variables categóricas y numéricas
categorical_cols = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# 5. One-Hot Encoding (ajustar únicamente con entrenamiento)
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_encoded = encoder.fit_transform(X_train[categorical_cols])
test_encoded = encoder.transform(X_test[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols)

# 6. Construir los DataFrames procesados
X_train = pd.concat([
    X_train[numeric_cols],
    pd.DataFrame(
        train_encoded,
        columns=encoded_cols,
        index=X_train.index
    )
], axis=1)

X_test = pd.concat([
    X_test[numeric_cols],
    pd.DataFrame(
        test_encoded,
        columns=encoded_cols,
        index=X_test.index
    )
], axis=1)



### Random Forest inputs

Keep the encoded features without scaling.

In [ ]:
# 7. Preparar datos para Random Forest (sin escalado)
X_train_rf = X_train.copy()
X_test_rf = X_test.copy()



### Logistic Regression inputs

Scale only `tenure`, `MonthlyCharges` and `TotalCharges`, fitting the scaler on training data.

In [ ]:
# 8. Preparar datos para Logistic Regression (con escalado)
X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

# Escalar únicamente variables numéricas continuas
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()

X_train_lr[scale_cols] = scaler.fit_transform(
    X_train_lr[scale_cols]
)

X_test_lr[scale_cols] = scaler.transform(
    X_test_lr[scale_cols]
)



### Check the prepared data

In [ ]:
# 9. Comprobaciones finales
print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)

print("\nColumnas coincidentes:", X_train.columns.equals(X_test.columns))

print("\nValores nulos:")
print("Train:", X_train.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print("\nDistribución de Churn:")
print("Train:")
print(y_train.value_counts(normalize=True).round(3))
print("Test:")
print(y_test.value_counts(normalize=True).round(3))

print("\nDatos preparados:")
print("Logistic Regression:", X_train_lr.shape, X_test_lr.shape)
print("Random Forest:", X_train_rf.shape, X_test_rf.shape)

## 4. Baseline models

Train Logistic Regression and Random Forest on their corresponding inputs. Evaluate both on the same test split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import pandas as pd

# 1. Definir modelos
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

# 2. Asociar los datos correspondientes a cada modelo
datasets = {
    "Logistic Regression": (X_train_lr, X_test_lr),
    "Random Forest": (X_train_rf, X_test_rf)
}

results = []



In [ ]:
# 3. Entrenar y evaluar
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, models.items()):

    X_tr, X_te = datasets[name]

    model.fit(X_tr, y_train)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        ax=ax,
        colorbar=False
    )

    ax.set_title(name)

plt.tight_layout()
plt.show()

# 4. Comparar métricas
results_df = pd.DataFrame(results).set_index("Modelo")

print("Comparación de modelos:")
display(results_df.round(4))

## 5. Random oversampling

Resample training data only. Refit clones of the baseline models and evaluate on the unchanged test data.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, ConfusionMatrixDisplay
)
import pandas as pd
import matplotlib.pyplot as plt

# 1. Aplicar oversampling exclusivamente al entrenamiento
ros = RandomOverSampler(random_state=42)

X_train_lr_os, y_train_lr_os = ros.fit_resample(
    X_train_lr, y_train
)

X_train_rf_os, y_train_rf_os = ros.fit_resample(
    X_train_rf, y_train
)

print("Distribución original:")
print(y_train.value_counts())

print("\nDistribución después de OS:")
print(y_train_lr_os.value_counts())



In [ ]:
# 2. Entrenar modelos con oversampling
models_os = {
    "Logistic Regression + OS": clone(models["Logistic Regression"]),
    "Random Forest + OS": clone(models["Random Forest"])
}

datasets_os = {
    "Logistic Regression + OS": (
        X_train_lr_os, X_test_lr, y_train_lr_os
    ),
    "Random Forest + OS": (
        X_train_rf_os, X_test_rf, y_train_rf_os
    )
}

results_os = []

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, models_os.items()):

    X_tr, X_te, y_tr = datasets_os[name]

    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results_os.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        ax=ax,
        colorbar=False
    )

    ax.set_title(name)

plt.tight_layout()
plt.show()

# 3. Comparar modelos originales y modelos con OS
results_os_df = pd.DataFrame(results_os).set_index("Modelo")

comparison_df = pd.concat([
    results_df,
    results_os_df
])

display(comparison_df.round(4))

## 6. Hyperparameter search

Select LR hyperparameters by five-fold stratified CV and F1. The pipeline fits the numeric scaler within each fold; categorical encoding was already fitted on the full training split.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
import pandas as pd

# 1. Configurar validación cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 2. Utilizar los datos codificados, pero sin escalar
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), scale_cols)
    ],
    remainder="passthrough"
)

# 3. Crear pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

# 4. Definir hiperparámetros
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

# 5. Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

# 6. Entrenar utilizando únicamente el conjunto de entrenamiento
grid_search.fit(X_train_rf, y_train)


In [ ]:

# 7. Mostrar mejores resultados
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print("\nMejor F1 promedio en validación:")
print(round(grid_search.best_score_, 4))

# 8. Tabla comparativa de configuraciones
cv_results = pd.DataFrame(grid_search.cv_results_)

comparison = cv_results[[
    "param_model__C",
    "param_model__class_weight",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_roc_auc"
]].copy()

comparison = comparison.sort_values(
    "mean_test_f1",
    ascending=False
)

display(comparison.round(4))

## 7. Threshold selection

Compare thresholds using out-of-fold training probabilities from the selected LR configuration. These reuse the model-selection folds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Recuperar el mejor modelo del GridSearchCV
best_model = clone(grid_search.best_estimator_)

# 2. Obtener probabilidades mediante validación cruzada
y_proba_cv = cross_val_predict(
    best_model,
    X_train_rf,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# 3. Evaluar diferentes thresholds
thresholds = np.round(np.arange(0.10, 0.91, 0.01), 2)

results_threshold = []

for threshold in thresholds:
    y_pred_cv = (y_proba_cv >= threshold).astype(int)

    results_threshold.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, y_pred_cv, zero_division=0
        ),
        "Recall": recall_score(
            y_train, y_pred_cv, zero_division=0
        ),
        "F1-Score": f1_score(
            y_train, y_pred_cv, zero_division=0
        )
    })

threshold_df = pd.DataFrame(results_threshold)

# 4. Seleccionar el threshold con mayor F1-Score
best_row = threshold_df.loc[
    threshold_df["F1-Score"].idxmax()
]

best_threshold = float(best_row["Threshold"])

print("Mejor threshold:", best_threshold)

print("\nMétricas con el threshold seleccionado:")
display(best_row.to_frame().T.round(4))


In [ ]:

# 5. Visualizar Precision, Recall y F1-Score
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    label="Precision"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    label="Recall"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1-Score"],
    label="F1-Score"
)

plt.axvline(
    x=best_threshold,
    color="red",
    linestyle="--",
    label=f"Mejor threshold: {best_threshold:.2f}"
)

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Optimización del threshold - Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 6. Mostrar los 10 mejores thresholds
print("\nLos 10 mejores thresholds según F1-Score:")

display(
    threshold_df.sort_values(
        by="F1-Score",
        ascending=False
    ).head(10).round(4)
)

## 8. Optimized model evaluation

Refit the selected LR on all training data and apply the original threshold of **0.55**. This comparison reuses the previously inspected test set.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)
import pandas as pd
import matplotlib.pyplot as plt

# 1. Recuperar el modelo optimizado
final_model = clone(grid_search.best_estimator_)

# 2. Entrenar con todo el conjunto de entrenamiento
final_model.fit(X_train_rf, y_train)

# 3. Obtener probabilidades sobre el test
y_proba_final = final_model.predict_proba(X_test_rf)[:, 1]

# 4. Aplicar el threshold seleccionado
final_threshold = 0.55
y_pred_final = (y_proba_final >= final_threshold).astype(int)

# 5. Calcular métricas
final_results = pd.DataFrame([{
    "Modelo": "Logistic Regression optimizada",
    "Accuracy": accuracy_score(y_test, y_pred_final),
    "Precision": precision_score(y_test, y_pred_final),
    "Recall": recall_score(y_test, y_pred_final),
    "F1-Score": f1_score(y_test, y_pred_final),
    "ROC-AUC": roc_auc_score(y_test, y_proba_final)
}]).set_index("Modelo")

# 6. Comparar con el modelo original
comparison_final = pd.concat([
    results_df.loc[["Logistic Regression"]],
    final_results
])

print("Comparación de resultados:")
display(comparison_final.round(4))

# 7. Mostrar matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final,
    display_labels=["No Churn", "Churn"],
    cmap="Blues",
    colorbar=False
)

plt.title("Logistic Regression optimizada - Threshold 0.55")
plt.tight_layout()
plt.show()